Pipeline preprocessing— MOFs Photocatalyse CO2
============================================


## ─────────────────────────────────────────────────────────────

In [3]:
# 0. IMPORTS & CONFIG
# ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

INPUT_PATH = r"D:\Mes projets\PFE\fastapi-ml\data\processed\dataset.csv"
OUTPUT_DIR = Path(r"D:\Mes projets\PFE\fastapi-ml\data\processed\pipeline_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## ─────────────────────────────────────────────────────────────

In [4]:
# 1. CHARGEMENT & AUDIT INITIAL
# ─────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(INPUT_PATH)
print(f"Shape brut : {df_raw.shape}")

# Colonnes à EXCLURE dès le début — métadonnées ou quasi-vides (>95% NaN)
COLS_META = [
    "refcode",                                            # identifiant non-informatif
    "doi",                                                # identifiant non-informatif
    "qualite.features_manquantes",                        # annotation interne
    "qualite.notes",                                      # annotation interne
    "proprietes_structurales.notes",                      # 99.7% NaN
    "conditions_reaction.pression_O2_bar",                # 99.4% NaN
    "performance_photocatalytique.co_production_rate_umol_g_h",  # 99.4% NaN
    "proprietes_electroniques.band_gap_ev",               # doublon de band_gap_eV (3 obs)
]

# Colonnes cibles 
CIBLES = [
    "performance.taux_production_umol_g_h",  
    "performance.selectivite_pct",           
    "performance.produit_majoritaire",        
]

# Colonnes à conserver mais NON utilisées comme features
CIBLES_DERIVEES = [
    "performance.type_taux",                  # annotation, pas feature
    "performance.AQE_pct",                    # trop sparse (84.5%) — output secondaire si besoin
    "performance.TON",                        # trop sparse (94.6%) — output secondaire si besoin
    "performance.H2_concurrent_umol_g_h",     # output secondaire si besoin
    "stabilite_photocatalytique.nombre_cycles",
    "stabilite_photocatalytique.duree_totale_h",
    "stabilite_photocatalytique.retention_pct",
    "stabilite_photocatalytique.caracterisation_post",
]

cols_to_drop = [c for c in COLS_META + CIBLES_DERIVEES if c in df_raw.columns]
df = df_raw.drop(columns=cols_to_drop).copy()


print(f"Shape après suppression colonnes meta/sparse : {df.shape}")
print(f"Colonnes restantes : {list(df.columns)}")

Shape brut : (316, 48)
Shape après suppression colonnes meta/sparse : (316, 32)
Colonnes restantes : ['mof.nom', 'mof.famille', 'mof.metal', 'mof.etat_oxydation', 'mof.ligand', 'mof.groupe_fonctionnel', 'mof.sites_metalliques_ouverts', 'mof.type_structure', 'cocatalyseur.type', 'cocatalyseur.charge_pct', 'conditions_reaction.source_lumiere', 'conditions_reaction.longueur_onde_nm', 'conditions_reaction.intensite_mW_cm2', 'conditions_reaction.agent_sacrificiel', 'conditions_reaction.solvant', 'conditions_reaction.concentration_catalyseur_g_L', 'conditions_reaction.pression_CO2_bar', 'conditions_reaction.temperature_C', 'conditions_reaction.duree_irradiation_h', 'conditions_reaction.atmosphere', 'proprietes_structurales.BET_m2_g', 'proprietes_structurales.volume_pores_cm3_g', 'proprietes_structurales.PLD_angstrom', 'proprietes_structurales.void_fraction', 'proprietes_electroniques.band_gap_eV', 'proprietes_electroniques.HOMO_eV_vs_NHE', 'proprietes_electroniques.LUMO_eV_vs_NHE', 'propriet

## ─────────────────────────────────────────────────────────────

In [5]:
# 2. CONSOLIDATION DES COLONNES DUPLIQUÉES
#    AVANT toute suppression ou imputation
# ─────────────────────────────────────────────────────────────────

# 2a. PLD : deux unités angstrom + nm — fusionner en une seule colonne angstrom
#     Règle : si angstrom présent → garder, sinon convertir nm × 10
PLD_A = "proprietes_structurales.PLD_angstrom"
PLD_N = "proprietes_structurales.PLD_nm"

if PLD_A in df.columns and PLD_N in df.columns:
    # Convertir nm → angstrom pour les lignes où seul PLD_nm est présent
    mask_only_nm = df[PLD_A].isna() & df[PLD_N].notna()
    df.loc[mask_only_nm, PLD_A] = df.loc[mask_only_nm, PLD_N] * 10.0
    df.drop(columns=[PLD_N], inplace=True)
    print(f"[PLD] Consolidé : {df[PLD_A].notna().sum()} valeurs (était 21+12 avant)")

# 2b. PLD encore très sparse (22/316 = 7%) → supprimer
#     Signal potentiel trop faible pour imputation (MNAR probable)
if PLD_A in df.columns:
    df.drop(columns=[PLD_A], inplace=True)
    print(f"[PLD] Supprimé (7% coverage — MNAR)")

# 2c. void_fraction : 97.2% NaN → supprimer
if "proprietes_structurales.void_fraction" in df.columns:
    df.drop(columns=["proprietes_structurales.void_fraction"], inplace=True)
    print("[void_fraction] Supprimé (97.2% NaN)")

print(f"\nShape après consolidation : {df.shape}")

[PLD] Consolidé : 22 valeurs (était 21+12 avant)
[PLD] Supprimé (7% coverage — MNAR)
[void_fraction] Supprimé (97.2% NaN)

Shape après consolidation : (316, 29)


## ─────────────────────────────────────────────────────────────

In [6]:
# 3. GESTION DES DOUBLONS
#    Les 52 lignes "dupliquées" doi+mof.nom correspondent à des
#    conditions différentes (agent sacrificiel, taux différent)
#    → Ce ne sont PAS des doublons au sens strict, mais des
#      re-mesures légitimes ou des entrées séparées.
#    → On garde TOUT sauf les vrais doublons identiques sur toutes
#      les colonnes pertinentes.
# ─────────────────────────────────────────────────────────────────
before = len(df)

# Vrais doublons stricts (toutes colonnes identiques)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Doublons stricts supprimés : {before - len(df)} | Restantes : {len(df)}")

# Note : on NE supprime PAS sur doi+mof.nom car des MOFs testés dans
# des conditions différentes sont des observations légitimes distinctes.

Doublons stricts supprimés : 0 | Restantes : 316


## ─────────────────────────────────────────────────────────────

In [7]:
# 4. NETTOYAGE DE LA CIBLE PRINCIPALE
# ─────────────────────────────────────────────────────────────────
TARGET = "performance.taux_production_umol_g_h"

# 4a. Supprimer les lignes où la cible ET la sélectivité ET le produit
#     sont tous NaN (lignes sans aucune information de performance)
mask_no_perf = (
    df[TARGET].isna() &
    df["performance.selectivite_pct"].isna() &
    df["performance.produit_majoritaire"].isna()
)
df = df[~mask_no_perf].reset_index(drop=True)
print(f"Lignes sans aucune performance supprimées : {mask_no_perf.sum()} | Restantes : {len(df)}")

# 4b. Correction de l'outlier index 256 (Ni-BDC)
#     Note qualite.notes : "104 mmol/g/h converti en µmol/g/h"
#     → Erreur de conversion : 104 mmol/g/h = 104 000 µmol/g/h (pas 104 000 000)
mask_nibdc = (df["mof.nom"] == "Ni-BDC") & (df[TARGET] > 1e7)
if mask_nibdc.sum() > 0:
    df.loc[mask_nibdc, TARGET] = 104_000.0
    print(f"[Outlier Ni-BDC] Corrigé : 1.04e8 → 104000 µmol/g/h")

# 4c. Vérification : outliers > 3 IQR dans l'espace log
log_taux = np.log1p(df[TARGET].dropna())
Q1, Q3 = log_taux.quantile(0.25), log_taux.quantile(0.75)
IQR = Q3 - Q1
outlier_mask = df[TARGET].notna() & (np.log1p(df[TARGET]) > Q3 + 3 * IQR)
print(f"\nOutliers log-space (> Q3+3IQR = {np.expm1(Q3 + 3*IQR):.0f} µmol/g/h) : {outlier_mask.sum()}")
if outlier_mask.sum() > 0:
    print(df.loc[outlier_mask, ["mof.nom", TARGET, "performance.type_taux"]].to_string())
    # On ne supprime PAS — on garde le flag pour traitement au niveau modèle
    df["flag_taux_extreme"] = outlier_mask.astype(int)
else:
    df["flag_taux_extreme"] = 0

# 4d. Log-transform de la cible
df["taux_log"] = np.log1p(df[TARGET])
print(f"\ntaux_log : skew={df['taux_log'].dropna().skew():.3f}, "
      f"mean={df['taux_log'].dropna().mean():.2f}, "
      f"std={df['taux_log'].dropna().std():.2f}")

# 4e. Agrégation classe produit
produit_map = {
    "CO":         "CO",
    "HCOOH":      "C1_liquide",
    "CH3OH":      "C1_liquide",
    "CH4":        "CH4",
    "H2":         "autre",
    "mixte":      "autre",
    "H2O2":       "autre",
    "C2H6":       "autre",
    "CH3CH2OH":   "autre",
    "C2H4":       "autre",
}
df["produit_classe"] = df["performance.produit_majoritaire"].map(produit_map)
print(f"\nproduit_classe distribution :")
print(df["produit_classe"].value_counts())

Lignes sans aucune performance supprimées : 8 | Restantes : 308
[Outlier Ni-BDC] Corrigé : 1.04e8 → 104000 µmol/g/h

Outliers log-space (> Q3+3IQR = 4265433 µmol/g/h) : 0

taux_log : skew=0.645, mean=3.70, std=2.31

produit_classe distribution :
produit_classe
CO            180
C1_liquide     55
CH4            36
autre          36
Name: count, dtype: int64


## ─────────────────────────────────────────────────────────────

In [8]:
# 5. STANDARDISATION DES VALEURS CATÉGORIELLES
# ─────────────────────────────────────────────────────────────────

# 5a. source_lumiere — normalisation des variantes
source_map = {
    "solaire_simule":  "solaire_simule",
    "solaire_simulé":  "solaire_simule",
    "xenon_35W":       "xenon",
    "xenon_150W":      "xenon",
    "xenon_300W":      "xenon",
    "xenon_500W":      "xenon",
    "xenon_1000W":     "xenon",
    "LED_420nm":       "LED",
    "LED_20W":         "LED",
    "Hg_400W":         "UV",
    "UV_visible":      "UV_visible",
    "UV":              "UV",
    "visible":         "visible",
}
df["source_lumiere"] = df["conditions_reaction.source_lumiere"].map(source_map)
df.drop(columns=["conditions_reaction.source_lumiere"], inplace=True)

# 5b. atmosphere — normalisation
atm_map = {
    "CO2_H2O":              "CO2_H2O",
    "CO2_pur":              "CO2_pur",
    "CO2_saturé_solution":  "CO2_sature",
    "CO2_sature_solution":  "CO2_sature",
    "CO2":                  "CO2_pur",
    "O2_pur":               "O2",
    "O2_H2O":               "O2",
}
df["atmosphere"] = df["conditions_reaction.atmosphere"].map(atm_map)
df.drop(columns=["conditions_reaction.atmosphere"], inplace=True)

print("Standardisation catégorielle effectuée.")
print(f"source_lumiere : {df['source_lumiere'].value_counts().to_dict()}")
print(f"atmosphere     : {df['atmosphere'].value_counts().to_dict()}")

Standardisation catégorielle effectuée.
source_lumiere : {'xenon': 189, 'solaire_simule': 53, 'visible': 51, 'UV': 8, 'LED': 6, 'UV_visible': 1}
atmosphere     : {'CO2_H2O': 154, 'CO2_pur': 139, 'CO2_sature': 12, 'O2': 3}


## ─────────────────────────────────────────────────────────────

In [9]:
# 6. IMPUTATION — RÈGLES MÉTIER (MCAR/MAR IDENTIFIÉS)
# ─────────────────────────────────────────────────────────────────

# ── 6a. Cocatalyseur ──────────────────────────────────────────
# "none" → "absent" (sémantique explicite)
df["cocatalyseur.type"] = df["cocatalyseur.type"].replace(
    {"none": "absent", "": "absent"}
).fillna("absent")

# charge_pct : 0 si absent (MCAR confirmé physiquement)
mask_absent = df["cocatalyseur.type"] == "absent"
df.loc[mask_absent, "cocatalyseur.charge_pct"] = 0.0

# charge_pct manquante avec cocatalyseur présent → médiane par type (MAR)
mask_present = df["cocatalyseur.type"] != "absent"
mask_nan_charge = df["cocatalyseur.charge_pct"].isna()
med_by_type = (
    df[mask_present & ~mask_nan_charge]
    .groupby("cocatalyseur.type")["cocatalyseur.charge_pct"]
    .median()
)
global_med_charge = df.loc[mask_present & ~mask_nan_charge, "cocatalyseur.charge_pct"].median()
df["flag.charge_inconnue"] = (mask_present & mask_nan_charge).astype(int)

for idx in df[mask_present & mask_nan_charge].index:
    cat = df.loc[idx, "cocatalyseur.type"]
    df.loc[idx, "cocatalyseur.charge_pct"] = (
        med_by_type[cat] if cat in med_by_type.index else global_med_charge
    )

print(f"[cocatalyseur] charge imputée : {df['flag.charge_inconnue'].sum()} lignes (médiane par type)")

# ── 6b. Conditions de réaction — valeurs physiques par défaut ──
# temperature : NaN → 25°C (température ambiante = défaut implicite dans les papiers)
df["conditions_reaction.temperature_C"] = pd.to_numeric(
    df["conditions_reaction.temperature_C"], errors="coerce"
).fillna(25.0)

# pression CO2 : NaN → 1.013 bar (pression atmosphérique)
df["conditions_reaction.pression_CO2_bar"] = pd.to_numeric(
    df["conditions_reaction.pression_CO2_bar"], errors="coerce"
).fillna(1.013)

# durée irradiation : NaN → médiane (MAR — pas de valeur physique par défaut)
med_duree = df["conditions_reaction.duree_irradiation_h"].median()
df["flag.duree_imputee"] = df["conditions_reaction.duree_irradiation_h"].isna().astype(int)
df["conditions_reaction.duree_irradiation_h"] = (
    df["conditions_reaction.duree_irradiation_h"].fillna(med_duree)
)
print(f"[durée] imputée à médiane ({med_duree:.1f}h) : {df['flag.duree_imputee'].sum()} lignes")

# concentration catalyseur : NaN → médiane (MAR)
med_conc = df["conditions_reaction.concentration_catalyseur_g_L"].median()
df["flag.conc_imputee"] = df["conditions_reaction.concentration_catalyseur_g_L"].isna().astype(int)
df["conditions_reaction.concentration_catalyseur_g_L"] = (
    df["conditions_reaction.concentration_catalyseur_g_L"].fillna(med_conc)
)

# intensite_mW_cm2 : 82.9% NaN — trop sparse pour imputer fiablement
# → flag binaire "intensité reportée" + suppression colonne
df["flag.intensite_reportee"] = df["conditions_reaction.intensite_mW_cm2"].notna().astype(int)
df.drop(columns=["conditions_reaction.intensite_mW_cm2"], inplace=True)
print("[intensité] convertie en flag binaire (82.9% NaN)")

# longueur_onde : 51.9% NaN
# → médiane par source_lumiere (MAR fort — la source détermine la longueur d'onde)
med_lo_by_source = (
    df.groupby("source_lumiere")["conditions_reaction.longueur_onde_nm"]
    .median()
)
df["flag.longueur_onde_imputee"] = df["conditions_reaction.longueur_onde_nm"].isna().astype(int)
df["conditions_reaction.longueur_onde_nm"] = df.apply(
    lambda row: (
        med_lo_by_source.get(row["source_lumiere"],
                              df["conditions_reaction.longueur_onde_nm"].median())
        if pd.isna(row["conditions_reaction.longueur_onde_nm"])
        else row["conditions_reaction.longueur_onde_nm"]
    ),
    axis=1,
)
print(f"[longueur_onde] imputée par source_lumiere : {df['flag.longueur_onde_imputee'].sum()} lignes")

# Cas résiduels longueur_onde (source_lumiere sans médiane calculable)
df["conditions_reaction.longueur_onde_nm"] = (
    df["conditions_reaction.longueur_onde_nm"]
    .fillna(df["conditions_reaction.longueur_onde_nm"].median())
)

# ── 6c. Groupes fonctionnels et solvant ──────────────────────
df["mof.groupe_fonctionnel"] = df["mof.groupe_fonctionnel"].fillna("absent").replace("", "absent")
df["conditions_reaction.solvant"] = df["conditions_reaction.solvant"].fillna("H2O")
df["conditions_reaction.agent_sacrificiel"] = (
    df["conditions_reaction.agent_sacrificiel"]
    .replace({"none": "absent", "": "absent"})
    .fillna("absent")
)

# ── 6d. mof.metal / etat_oxydation ──────────────────────────
df["mof.metal"] = df["mof.metal"].fillna("inconnu")

# etat_oxydation : cast propre + valeur "inconnue" pour NaN
df["mof.etat_oxydation"] = (
    df["mof.etat_oxydation"]
    .astype(str)
    .replace({"nan": "inconnu", "NaN": "inconnu"})
)

# mof.ligand
df["mof.ligand"] = df["mof.ligand"].fillna("inconnu")

# sites_metalliques_ouverts : 54.4% NaN → flag binaire + supprimer
df["flag.sites_ouverts_connu"] = df["mof.sites_metalliques_ouverts"].notna().astype(int)
df.drop(columns=["mof.sites_metalliques_ouverts"], inplace=True)

print("\nImputation conditions / MOF terminée.")

[cocatalyseur] charge imputée : 48 lignes (médiane par type)
[durée] imputée à médiane (4.0h) : 82 lignes
[intensité] convertie en flag binaire (82.9% NaN)
[longueur_onde] imputée par source_lumiere : 158 lignes

Imputation conditions / MOF terminée.


## ─────────────────────────────────────────────────────────────

In [10]:
# 7. IMPUTATION DES PROPRIÉTÉS STRUCTURALES & ELECTRONIQUES
# ─────────────────────────────────────────────────────────────────

# ── 7a. BET ─────────────────────────────────────────────────
# 58.2% NaN — MAR partiel (corrélé à mof.famille)
BET_COL = "proprietes_structurales.BET_m2_g"
df["flag.BET_imputed"] = df[BET_COL].isna().astype(int)

# Médiane par famille (MAR — les familles ont des surfaces typiques)
med_bet_fam = df.groupby("mof.famille")[BET_COL].median()
df[BET_COL] = df.apply(
    lambda row: (
        med_bet_fam.get(row["mof.famille"], df[BET_COL].median())
        if pd.isna(row[BET_COL]) else row[BET_COL]
    ),
    axis=1,
)
# Fallback global pour familles sans médiane
df[BET_COL] = df[BET_COL].fillna(df[BET_COL].median())
print(f"[BET] imputé par médiane/famille : {df['flag.BET_imputed'].sum()} lignes")

# volume_pores : 84.8% NaN — trop sparse, flag + suppression
df["flag.volume_pores_connu"] = df["proprietes_structurales.volume_pores_cm3_g"].notna().astype(int)
df.drop(columns=["proprietes_structurales.volume_pores_cm3_g"], inplace=True)

# ── 7b. Band Gap ─────────────────────────────────────────────
# NOTE IMPORTANTE : HOMO et LUMO NE SONT PAS supprimés ici.
# Analyse : HOMO a une corrélation indépendante avec le taux (r=-0.29),
# différente du band_gap (r=-0.25). HOMO + band_gap = information complémentaire.
# Le manque de couverture (31% pour HOMO) est géré par imputation séparée.

BAND_GAP_COL = "proprietes_electroniques.band_gap_eV"
ABS_VIS_COL  = "proprietes_electroniques.absorption_visible"

# Encoder absorption_visible
df[ABS_VIS_COL] = df[ABS_VIS_COL].map(
    {True: 1, False: 0, "True": 1, "False": 0}
)
# NaN absorption_visible → 1 (semi-conducteur actif par défaut dans ce corpus)
df[ABS_VIS_COL] = df[ABS_VIS_COL].fillna(1.0)
df["flag.band_gap_imputed"] = df[BAND_GAP_COL].isna().astype(int)
df["flag.band_gap_method"]  = "observed"

# Cas MCAR : absorption_visible = False → band_gap non pertinent → 10 eV (isolant)
mask_mcar = (df[ABS_VIS_COL] == 0) & df[BAND_GAP_COL].isna()
df.loc[mask_mcar, BAND_GAP_COL]          = 10.0
df.loc[mask_mcar, "flag.band_gap_method"] = "mcar_10ev"

# Cas MAR : absorption_visible = True → MICE sur sous-ensemble semi-conducteurs
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import OrdinalEncoder

mask_mar = (df[ABS_VIS_COL] == 1) & df[BAND_GAP_COL].isna()

if mask_mar.sum() > 0:
    type_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    df["__type_enc"] = type_enc.fit_transform(df[["mof.type_structure"]])

    mice_cols = [BAND_GAP_COL, ABS_VIS_COL,
                 "conditions_reaction.longueur_onde_nm", "__type_enc"]
    mask_active = df[ABS_VIS_COL] == 1
    subset = df.loc[mask_active, mice_cols].copy()

    n_obs = subset[BAND_GAP_COL].notna().sum()
    if n_obs >= 10:
        imp = IterativeImputer(
            estimator=BayesianRidge(), max_iter=20,
            random_state=42, min_value=0.5, max_value=9.0
        )
        imputed = imp.fit_transform(subset)
        imputed_df = pd.DataFrame(imputed, columns=mice_cols, index=subset.index)
        df.loc[mask_mar, BAND_GAP_COL]          = imputed_df.loc[mask_mar, BAND_GAP_COL]
        df.loc[mask_mar, "flag.band_gap_method"] = "mar_mice_bayesian"
    else:
        fallback = df.loc[mask_active & df[BAND_GAP_COL].notna(), BAND_GAP_COL].median()
        df.loc[mask_mar, BAND_GAP_COL]          = fallback
        df.loc[mask_mar, "flag.band_gap_method"] = "mar_fallback_median"

    df.drop(columns=["__type_enc"], inplace=True)

# Cas résiduels (absorption_visible NaN)
mask_resid = df[BAND_GAP_COL].isna()
if mask_resid.sum() > 0:
    fallback = df.loc[df["flag.band_gap_method"] == "observed", BAND_GAP_COL].median()
    df.loc[mask_resid, BAND_GAP_COL]          = fallback
    df.loc[mask_resid, "flag.band_gap_method"] = "residuel_median"

print(f"[band_gap] distribution méthodes :")
print(df["flag.band_gap_method"].value_counts())

# ── 7c. HOMO — imputation séparée (ne pas mélanger avec band_gap) ──
HOMO_COL = "proprietes_electroniques.HOMO_eV_vs_NHE"
LUMO_COL = "proprietes_electroniques.LUMO_eV_vs_NHE"

# HOMO : 69% NaN — imputer par médiane par famille × type_structure
df["flag.HOMO_imputed"] = df[HOMO_COL].isna().astype(int)
med_homo = df.groupby(["mof.famille", "mof.type_structure"])[HOMO_COL].median()

def impute_homo(row):
    if pd.notna(row[HOMO_COL]):
        return row[HOMO_COL]
    key = (row["mof.famille"], row["mof.type_structure"])
    if key in med_homo.index and pd.notna(med_homo[key]):
        return med_homo[key]
    return df[HOMO_COL].median()

df[HOMO_COL] = df.apply(impute_homo, axis=1)
print(f"[HOMO] imputé : {df['flag.HOMO_imputed'].sum()} lignes (médiane famille×structure)")

# LUMO : 59.2% NaN — relation physique LUMO ≈ HOMO - band_gap
# C'est la relation thermodynamique la plus fiable
df["flag.LUMO_imputed"] = df[LUMO_COL].isna().astype(int)
mask_lumo_nan = df[LUMO_COL].isna()
# LUMO = HOMO - band_gap (convention NHE)
df.loc[mask_lumo_nan, LUMO_COL] = (
    df.loc[mask_lumo_nan, HOMO_COL] - df.loc[mask_lumo_nan, BAND_GAP_COL]
)
print(f"[LUMO] imputé via HOMO-band_gap : {df['flag.LUMO_imputed'].sum()} lignes")

[BET] imputé par médiane/famille : 183 lignes
[band_gap] distribution méthodes :
flag.band_gap_method
mar_mice_bayesian    154
observed             138
mcar_10ev             16
Name: count, dtype: int64
[HOMO] imputé : 212 lignes (médiane famille×structure)
[LUMO] imputé via HOMO-band_gap : 181 lignes


## ─────────────────────────────────────────────────────────────

In [11]:
# 8. ENCODAGE DES VARIABLES CATÉGORIELLES
# ─────────────────────────────────────────────────────────────────
# Stratégie : Target Encoding pour les catégorielles à forte cardinalité
# One-Hot pour les catégorielles à faible cardinalité (≤ 5 classes)
# ATTENTION : le target encoding sera refitté dans chaque fold CV en production
#             Ici on utilise une version globale pour l'analyse exploratoire

TARGET_LOG = "taux_log"

# Colonnes catégorielles après nettoyage
cat_high_card = [
    "mof.famille",            # 11 → target enc
    "mof.metal",              # 50+ → target enc
    "mof.ligand",             # 47  → target enc
    "mof.groupe_fonctionnel", # 16  → target enc
    "cocatalyseur.type",      # 23  → target enc
    "conditions_reaction.agent_sacrificiel",  # 13 → target enc
    "conditions_reaction.solvant",            # 9  → target enc
    "source_lumiere",                         # 6  → target enc
]

cat_low_card = [
    "mof.type_structure",    # 4 classes → one-hot
    "mof.etat_oxydation",    # 5 classes → one-hot
    "atmosphere",            # 5 classes → one-hot
]

# Target encoding (moyenne de taux_log par modalité)
mask_target_known = df[TARGET_LOG].notna()

for col in cat_high_card:
    if col not in df.columns:
        continue
    enc_map = df.loc[mask_target_known].groupby(col)[TARGET_LOG].mean()
    global_mean = df.loc[mask_target_known, TARGET_LOG].mean()
    df[f"{col}_enc"] = df[col].map(enc_map).fillna(global_mean)

# One-hot encoding
df = pd.get_dummies(df, columns=cat_low_card, drop_first=False, dtype=int)

print("Encodage terminé.")
print(f"Shape après encodage : {df.shape}")

# Supprimer les colonnes originales catégorielles hautes cardinalités
df.drop(columns=[c for c in cat_high_card if c in df.columns], inplace=True)

# Supprimer mof.nom (identifiant) et autres métadonnées textuelles résiduelles
for c in ["mof.nom", "performance.produit_majoritaire",
          "performance.taux_production_umol_g_h",
          "flag.band_gap_method"]:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

Encodage terminé.
Shape après encodage : (308, 60)


## ─────────────────────────────────────────────────────────────

In [12]:
# 9. FEATURE ENGINEERING — INTERACTIONS PHYSIQUES
#    UNIQUEMENT des interactions justifiées par la chimie
# ─────────────────────────────────────────────────────────────────
BG = "proprietes_electroniques.band_gap_eV"

# 9a. Potentiel de réduction CO2 (ΔE vs LUMO)
# CO2 → CO : E° = -0.53 V vs NHE ; CO2 → CH4 : E° = -0.24 V vs NHE
# ΔLUMO_CO = LUMO - (-0.53) : marge de réduction → plus c'est négatif, meilleur
LUMO = "proprietes_electroniques.LUMO_eV_vs_NHE"
df["delta_LUMO_CO2"] = df[LUMO] - (-0.53)   # positif = favorable
df["delta_LUMO_CH4"] = df[LUMO] - (-0.24)

# 9b. BET × durée → proxy d'exposition totale (surface active × temps)
BET = "proprietes_structurales.BET_m2_g"
DUR = "conditions_reaction.duree_irradiation_h"
df["BET_x_duree"] = df[BET] * df[DUR]

# 9c. Indicateur cocatalyseur présent
df["has_cocatalyseur"] = (df["cocatalyseur.charge_pct"] > 0).astype(int)

print("Features d'interaction créées : delta_LUMO_CO2, delta_LUMO_CH4, BET_x_duree, has_cocatalyseur")

Features d'interaction créées : delta_LUMO_CO2, delta_LUMO_CH4, BET_x_duree, has_cocatalyseur


## ─────────────────────────────────────────────────────────────

In [13]:
# 10. STANDARDISATION
#     PRINCIPE CLEF : on ne scale PAS les cibles MTL
#     On ne scale PAS les features binaires (flags, one-hot)
#     RobustScaler pour toutes les features continues
#     FIT uniquement sur les observations NON-imputées quand possible
# ─────────────────────────────────────────────────────────────────
from sklearn.preprocessing import RobustScaler
import joblib

SCALER_DIR = Path("D:/Mes projets/PFE/fastapi-ml/models/scalers")
SCALER_DIR.mkdir(parents=True, exist_ok=True)

CIBLES_FINALES = [TARGET_LOG, "performance.selectivite_pct", "produit_classe"]
CIBLES_FINALES = [c for c in CIBLES_FINALES if c in df.columns]

# Colonnes binaires — NE PAS scaler
BINAIRES = [c for c in df.columns if (
    c.startswith("flag") or
    c.startswith("mof.type_structure_") or
    c.startswith("mof.etat_oxydation_") or
    c.startswith("atmosphere_") or
    c in ["proprietes_electroniques.absorption_visible",
          "has_cocatalyseur", "flag_taux_extreme"]
)]

# Colonnes target-encoded — à scaler DANS les folds (ici global pour analyse)
TARGET_ENC_COLS = [c for c in df.columns if c.endswith("_enc")]

# Features continues — à scaler globalement
CONTINUOUS_COLS = [
    c for c in df.columns
    if c not in CIBLES_FINALES + BINAIRES + TARGET_ENC_COLS
    and df[c].dtype in [np.float64, np.int64, float, int]
    and df[c].nunique() > 2
]

print(f"Colonnes continues à scaler : {len(CONTINUOUS_COLS)}")
print(f"Colonnes target-encoded     : {len(TARGET_ENC_COLS)}")
print(f"Colonnes binaires           : {len(BINAIRES)}")

# Scaler BET — fit sur observées seulement
BET_OBS_MASK = df["flag.BET_imputed"] == 0
scaler_BET = RobustScaler()
scaler_BET.fit(df.loc[BET_OBS_MASK, [BET]].values.reshape(-1, 1))
df[BET] = scaler_BET.transform(df[[BET]].values.reshape(-1, 1))
joblib.dump(scaler_BET, SCALER_DIR / "scaler_BET.pkl")

# Scaler global pour autres features continues
other_cont = [c for c in CONTINUOUS_COLS if c != BET]
if other_cont:
    scaler_cont = RobustScaler()
    df[other_cont] = scaler_cont.fit_transform(df[other_cont])
    joblib.dump(scaler_cont, SCALER_DIR / "scaler_continuous.pkl")

# Scaler target-encoded — global (à refitter par fold en production)
if TARGET_ENC_COLS:
    scaler_te = RobustScaler()
    df[TARGET_ENC_COLS] = scaler_te.fit_transform(df[TARGET_ENC_COLS])
    # Clip [-3, 3] après scaling pour éviter la compression des autres features
    df[TARGET_ENC_COLS] = df[TARGET_ENC_COLS].clip(-3, 3)
    joblib.dump(scaler_te, SCALER_DIR / "scaler_target_enc.pkl")

# Clip des features continues (pas binaires, pas cibles)
cols_to_clip = other_cont + [BET]
df[cols_to_clip] = df[cols_to_clip].clip(-3, 3)

print("\nStandardisation terminée.")

Colonnes continues à scaler : 13
Colonnes target-encoded     : 8
Colonnes binaires           : 28

Standardisation terminée.


## ─────────────────────────────────────────────────────────────

In [14]:
# 11. VALIDATION DU DATASET FINAL
# ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("RAPPORT DE VALIDATION FINAL")
print("=" * 65)

features_only = df.drop(columns=CIBLES_FINALES, errors="ignore")

# NaN restants
nan_features = features_only.isna().sum()
nan_features = nan_features[nan_features > 0]
if len(nan_features) == 0:
    print("\n✅ Aucun NaN dans les features")
else:
    print("\n⚠️  NaN résiduels dans features :")
    print(nan_features)

nan_cibles = {c: df[c].isna().sum() for c in CIBLES_FINALES if c in df.columns}
print(f"\nNaN cibles MTL (attendus — mask) :")
for c, n in nan_cibles.items():
    pct = n / len(df) * 100
    print(f"  {c:<45} {n:>3} NaN ({pct:.0f}%)")

# Ranges
print(f"\n{'Colonne':<50} {'mean':>7} {'std':>7} {'min':>7} {'max':>7}")
print("-" * 75)
for col in features_only.select_dtypes(include="number").columns:
    s = df[col]
    warn = " ⚠️" if (s.std() > 3.5 or abs(s.mean()) > 2) else ""
    print(f"{col:<50} {s.mean():>7.3f} {s.std():>7.3f} {s.min():>7.3f} {s.max():>7.3f}{warn}")

print(f"\nShape final : {df.shape}")


RAPPORT DE VALIDATION FINAL

✅ Aucun NaN dans les features

NaN cibles MTL (attendus — mask) :
  taux_log                                       45 NaN (15%)
  performance.selectivite_pct                   211 NaN (69%)
  produit_classe                                  1 NaN (0%)

Colonne                                               mean     std     min     max
---------------------------------------------------------------------------
cocatalyseur.charge_pct                              0.438   0.762   0.000   3.000
conditions_reaction.longueur_onde_nm                -0.614   1.260  -3.000   3.000
conditions_reaction.concentration_catalyseur_g_L     0.213   0.654  -0.390   3.000
conditions_reaction.pression_CO2_bar                -0.234   1.085  -3.000   3.000
conditions_reaction.temperature_C                    0.097   1.345  -3.000   3.000
conditions_reaction.duree_irradiation_h              0.325   1.380  -3.000   3.000
proprietes_structurales.BET_m2_g                     0.277   

## ─────────────────────────────────────────────────────────────

In [15]:
# 12. SAUVEGARDE
# ─────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_DIR / "dataset_clean_final.csv", index=False)
print(f"\n✅ Dataset sauvegardé → {OUTPUT_DIR / 'dataset_clean_final.csv'}")
print(f"   {df.shape[0]} lignes × {df.shape[1]} colonnes")


✅ Dataset sauvegardé → D:\Mes projets\PFE\fastapi-ml\data\processed\pipeline_outputs\dataset_clean_final.csv
   308 lignes × 52 colonnes


In [16]:
# 12. SAUVEGARDE DU PIPELINE AVEC MLFLOW (APRES FEATURE ENGINEERING)

import mlflow
import json
import re
from pathlib import Path

# Configuration MLflow directe (sans import)
MLFLOW_TRACKING_URI = "D:/Mes projets/PFE/fastapi-ml/mlflow_runs"
MLFLOW_EXPERIMENT_NAME = "MOF_photocatalysis_pipeline"

Path(MLFLOW_TRACKING_URI).mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"file:///{MLFLOW_TRACKING_URI}")

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    mlflow.create_experiment(
        MLFLOW_EXPERIMENT_NAME,
        artifact_location=f"file:///{MLFLOW_TRACKING_URI}/{MLFLOW_EXPERIMENT_NAME}"
    )
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(f"MLflow configure avec l'experience : {MLFLOW_EXPERIMENT_NAME}")
print(f"Tracking URI : {MLFLOW_TRACKING_URI}")

# Fonction pour nettoyer les cles
def clean_param_key(key):
    replacements = {
        '%': 'pct', '$': 'dollar', '@': 'at', '?': 'q',
        '=': 'eq', ';': '', ':': '', '[': '_', ']': '_',
        '(': '_', ')': '_', '{': '_', '}': '_', '|': '_',
        '\\': '_', '/': '_', '+': '_plus_', '*': '_star_',
        '&': '_and_', '#': '_sharp_',
    }
    for old, new in replacements.items():
        key = key.replace(old, new)
    key = re.sub(r'[^a-zA-Z0-9_\-\.\/\s]', '_', key)
    if len(key) > 250:
        key = key[:250]
    return key

SCALER_DIR = Path("D:/Mes projets/PFE/fastapi-ml/models/scalers")



# Demarrer un run MLflow
with mlflow.start_run(run_name="feature_engineering") as run:
    
    print(f"Demarrage du run MLflow : {run.info.run_id}")
    
    # 1. Log des parametres
    mlflow.log_param("feature_count", len(df.columns))
    mlflow.log_param("row_count", len(df))
    mlflow.log_param("target_columns", str(CIBLES_FINALES))
    mlflow.log_param("binary_columns_count", len(BINAIRES))
    mlflow.log_param("continuous_columns_count", len(CONTINUOUS_COLS))
    mlflow.log_param("target_enc_columns_count", len(TARGET_ENC_COLS))
    
    # 2. Log des metriques
    mlflow.log_metric("missing_rate_target_log", float(df["taux_log"].isna().mean()))
    mlflow.log_metric("missing_rate_selectivite", float(df["performance.selectivite_pct"].isna().mean()))
    
    # 3. Sauvegarder les scalers
    if SCALER_DIR.exists():
        for scaler_file in SCALER_DIR.glob("*.pkl"):
            mlflow.log_artifact(str(scaler_file))
            print(f"  Artefact : {scaler_file.name}")
    
    # 4. Sauvegarder les mappings
    mappings = {
        "cibles_finales": CIBLES_FINALES,
        "colonnes_binaires": BINAIRES,
        "colonnes_target_enc": TARGET_ENC_COLS,
        "colonnes_continues": CONTINUOUS_COLS,
    }
    mappings_path = SCALER_DIR / "preprocessing_mappings.json"
    with open(mappings_path, "w") as f:
        json.dump(mappings, f, indent=2)
    mlflow.log_artifact(str(mappings_path))
    
    # 5. Tags
    mlflow.set_tag("pipeline_stage", "feature_engineering")
    mlflow.set_tag("data_version", "clean_final")
    
    print(f"Run termine. Run ID : {run.info.run_id}")

MLflow configure avec l'experience : MOF_photocatalysis_pipeline
Tracking URI : D:/Mes projets/PFE/fastapi-ml/mlflow_runs
Demarrage du run MLflow : 86b5ba95356f45c48883c177704a235f
  Artefact : scaler_BET.pkl
  Artefact : scaler_continuous.pkl
  Artefact : scaler_target_enc.pkl
Run termine. Run ID : 86b5ba95356f45c48883c177704a235f
